# Notebook 21: Neural Criticality & Avalanche Analysis in NeuroFMx
**Testing for Brain-Like Self-Organized Criticality**

## What You'll Learn

In this advanced notebook, you'll test whether NeuroFMx exhibits critical dynamics:

- **Neuronal Avalanche Detection**: Identify cascading activation patterns
- **Power Law Analysis**: Test for scale-free size and duration distributions
- **Branching Parameter (σ)**: Estimate distance from critical point (σ = 1)
- **Self-Organized Criticality**: Test SOC hypotheses in foundation models
- **Distance from Criticality (DCC)**: Quantify deviation from optimal regime
- **NeuroFMx Integration**: Analyze real model activations for criticality
- **Layer-Wise Criticality**: Track how criticality emerges across depth

**Prerequisites**: 
- Notebooks 01-20 (mechanistic interpretability foundation)
- Understanding of power laws and scale-free systems
- Python 3.8+, PyTorch, Bokeh

**Time Estimate**: 45-60 minutes

---

## Why Criticality Matters

### The Criticality Hypothesis

**Biological brains operate near a critical point** between:
- **Subcritical** (σ < 1): Activity dies out, poor information transmission
- **Supercritical** (σ > 1): Activity explodes, seizure-like states
- **Critical** (σ ≈ 1): Optimal balance, maximal information processing

### Benefits of Critical Dynamics

1. **Maximal Dynamic Range**: Respond to stimuli across many scales
2. **Efficient Information Transmission**: Power-law correlations
3. **Computational Power**: Edge-of-chaos computation
4. **Sensitivity & Stability**: Balance responsiveness and robustness

### Testing NeuroFMx for Criticality

**Key Questions**:
- Do activations exhibit neuronal avalanches?
- Are avalanche sizes and durations power-law distributed?
- Is the branching parameter σ ≈ 1?
- Does NeuroFMx self-organize toward criticality during training?

**Hypothesis**: NeuroFMx's multi-rate Mamba architecture may naturally produce critical dynamics due to:
- Recurrent state-space dynamics (feedback loops)
- Multi-scale processing (cross-scale coupling)
- Normalization layers (activity regulation)

---

## Key Papers

- **Beggs & Plenz (2003)**: Neuronal avalanches in cortical networks
- **Shew et al. (2009)**: Information capacity at criticality
- **Chialvo (2010)**: Emergent complex neural dynamics
- **Hahn et al. (2017)**: Spontaneous cortical activity at criticality

---

In [ ]:
# Imports
import sys
import warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from typing import Dict, List, Optional, Tuple
from scipy.stats import linregress, kstest, powerlaw
from scipy.optimize import curve_fit
warnings.filterwarnings('ignore')

# Bokeh for interactive visualization
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import column, row, gridplot
from bokeh.models import HoverTool, ColorBar, LinearColorMapper, ColumnDataSource, Span
from bokeh.palettes import Viridis256, Spectral11, RdYlGn11, Category20, Turbo256
from bokeh.transform import linear_cmap
import pandas as pd

# Enable Bokeh in notebook
output_notebook()

# Criticality analysis tools
from neuros_mechint.fractals import (
    NeuronalAvalanche,
    BranchingProcess,
    CriticalityDetector,
    SelfOrganizedCriticality
)

# NeuroFMx model
try:
    from neuros_neurofm.models import NeuroFMX
    NEUROFMX_AVAILABLE = True
except ImportError:
    NEUROFMX_AVAILABLE = False
    print("⚠ NeuroFMx not available. Using simulated activations.")

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Bokeh interactive plotting: ✓")
print(f"NeuroFMx available: {'✓' if NEUROFMX_AVAILABLE else '✗ (using simulation)'}")

---

## Part 1: Generate Critical Activity with Branching Process

First, create ground-truth critical dynamics for validation.

---

In [ ]:
# Generate branching process at different σ values
def generate_branching_process(n_neurons=200, n_timesteps=5000, sigma=1.0):
    """
    Generate branching process.
    
    σ < 1: Subcritical (activity dies)
    σ = 1: Critical (scale-free avalanches)
    σ > 1: Supercritical (activity explodes)
    """
    activity = np.zeros((n_timesteps, n_neurons))
    
    # Initialize with some seed activity
    activity[0, np.random.choice(n_neurons, size=10, replace=False)] = 1
    
    for t in range(1, n_timesteps):
        active_neurons = np.where(activity[t-1] > 0)[0]
        
        for neuron in active_neurons:
            # Each active neuron activates ~sigma neighbors
            n_offspring = np.random.poisson(sigma)
            
            if n_offspring > 0:
                targets = np.random.choice(n_neurons, size=min(n_offspring, n_neurons), replace=False)
                activity[t, targets] = 1
        
        # Random spontaneous activations (external drive)
        if np.random.rand() < 0.02:  # 2% chance
            activity[t, np.random.choice(n_neurons, size=5, replace=False)] = 1
    
    return activity

# Generate activity at critical point
print("Generating branching process at criticality (σ=1.0)...\n")
n_neurons = 200
n_timesteps = 5000
sigma_critical = 1.0

critical_activity = generate_branching_process(n_neurons, n_timesteps, sigma_critical)

print(f"Generated activity:")
print(f"  Shape: {critical_activity.shape}")
print(f"  Total spikes: {critical_activity.sum():.0f}")
print(f"  Mean activity: {critical_activity.mean():.4f}")
print(f"  Active timesteps: {(critical_activity.sum(axis=1) > 0).sum()} / {n_timesteps}")

In [ ]:
# Visualize raster plot
print("Creating interactive raster plot...\n")

# Plot first 1000 timesteps
window = 1000
activity_subset = critical_activity[:window, :]

# Get spike times for each neuron
spike_times = []
neuron_ids = []

for neuron_idx in range(n_neurons):
    times = np.where(activity_subset[:, neuron_idx] > 0)[0]
    spike_times.extend(times)
    neuron_ids.extend([neuron_idx] * len(times))

# Create raster plot
p1 = figure(
    title=f'Neural Activity Raster (σ={sigma_critical}, first {window} timesteps)',
    width=1000,
    height=400,
    x_axis_label='Time',
    y_axis_label='Neuron ID',
    toolbar_location='above'
)

p1.circle(spike_times, neuron_ids, size=2, color='black', alpha=0.6)

# Add activity level trace
activity_trace = activity_subset.sum(axis=1)
p2 = figure(
    title='Population Activity',
    width=1000,
    height=200,
    x_axis_label='Time',
    y_axis_label='Active Neurons',
    toolbar_location='above'
)

p2.line(np.arange(window), activity_trace, color='steelblue', line_width=2, alpha=0.8)

# Arrange plots
layout = column(p1, p2)
show(layout)

print("✓ Interactive raster plot created")

---

## Part 2: Detect Neuronal Avalanches

Identify spatiotemporal cascades of activity.

**Definition**: An avalanche is a contiguous period of activity separated by silent periods.

---

In [ ]:
# Detect avalanches
print("Detecting neuronal avalanches...\n")

detector = NeuronalAvalanche(threshold=0.0)
avalanches = detector.detect_avalanches(critical_activity)

print(f"Detected {len(avalanches)} avalanches")

# Extract statistics
avalanche_sizes = np.array([av['size'].sum() for av in avalanches if av['size'].sum() > 0])
avalanche_durations = np.array([av['duration'] for av in avalanches if av['duration'] > 0])

print(f"\nAvalanche Statistics:")
print(f"  Mean size: {avalanche_sizes.mean():.2f} spikes")
print(f"  Max size: {avalanche_sizes.max():.0f} spikes")
print(f"  Mean duration: {avalanche_durations.mean():.2f} timesteps")
print(f"  Max duration: {avalanche_durations.max():.0f} timesteps")

# Show first 5 avalanches
print(f"\nFirst 5 avalanches:")
for i, av in enumerate(avalanches[:5]):
    print(f"  {i+1}. Start: {av['start']:4d}, Duration: {av['duration']:3d}, Size: {av['size'].sum():4.0f}")

---

## Part 3: Power Law Analysis

At criticality, avalanche sizes follow: **P(s) ∝ s^(-τ)** with τ ≈ 1.5

---

In [ ]:
# Fit power law to avalanche sizes
print("Fitting power law to avalanche sizes...\n")

# Use logarithmic binning for better power law fitting
sizes_nonzero = avalanche_sizes[avalanche_sizes > 0]

# Method 1: Log-log linear regression
log_sizes = np.log10(sizes_nonzero)
counts, bins = np.histogram(log_sizes, bins=25)
counts = counts[counts > 0]
bin_centers = (bins[:-1] + bins[1:])[len(bins)-len(counts)-1:len(bins)-1] / 2
log_counts = np.log10(counts)

# Linear fit in log-log space
slope, intercept, r_value, p_value, std_err = linregress(bin_centers, log_counts)
tau = -slope  # Power law exponent

print(f"Power Law Fit Results:")
print(f"  Exponent τ: {tau:.3f} (theory: ~1.5)")
print(f"  R²: {r_value**2:.4f}")
print(f"  p-value: {p_value:.4e}")
print(f"  Std error: {std_err:.4f}")

# Method 2: Maximum likelihood (optional)
try:
    # Fit power law using powerlaw package approach
    alpha_ml = 1 + len(sizes_nonzero) / np.sum(np.log(sizes_nonzero / sizes_nonzero.min()))
    print(f"\nMaximum Likelihood Estimate:")
    print(f"  Exponent α: {alpha_ml:.3f}")
except:
    pass

In [ ]:
# Interactive power law visualization
print("Creating interactive power law visualization...\n")

# Plot 1: Size distribution (log-log)
unique_sizes, size_counts = np.unique(sizes_nonzero, return_counts=True)

p1 = figure(
    title='Avalanche Size Distribution (Power Law)',
    width=600,
    height=500,
    x_axis_type='log',
    y_axis_type='log',
    x_axis_label='Avalanche Size (spikes)',
    y_axis_label='Count',
    toolbar_location='above'
)

# Data points
p1.circle(unique_sizes, size_counts, size=8, color='steelblue', alpha=0.6, 
          legend_label='Data')

# Power law fit
x_fit = np.logspace(np.log10(unique_sizes.min()), np.log10(unique_sizes.max()), 100)
y_fit = 10**intercept * x_fit**(-tau)
p1.line(x_fit, y_fit, color='red', line_width=3, line_dash='dashed',
        legend_label=f'Power Law (τ={tau:.2f}, R²={r_value**2:.3f})')

p1.legend.location = "top_right"

# Plot 2: Duration distribution
durations_nonzero = avalanche_durations[avalanche_durations > 0]
unique_durations, duration_counts = np.unique(durations_nonzero, return_counts=True)

# Fit duration power law
log_dur = np.log10(durations_nonzero)
dur_counts, dur_bins = np.histogram(log_dur, bins=20)
dur_counts = dur_counts[dur_counts > 0]
dur_centers = (dur_bins[:-1] + dur_bins[1:])[len(dur_bins)-len(dur_counts)-1:len(dur_bins)-1] / 2
log_dur_counts = np.log10(dur_counts)

slope_dur, intercept_dur, r_dur, _, _ = linregress(dur_centers, log_dur_counts)
tau_dur = -slope_dur

p2 = figure(
    title='Avalanche Duration Distribution',
    width=600,
    height=500,
    x_axis_type='log',
    y_axis_type='log',
    x_axis_label='Duration (timesteps)',
    y_axis_label='Count',
    toolbar_location='above'
)

p2.circle(unique_durations, duration_counts, size=8, color='coral', alpha=0.6,
          legend_label='Data')

x_fit_dur = np.logspace(np.log10(unique_durations.min()), np.log10(unique_durations.max()), 100)
y_fit_dur = 10**intercept_dur * x_fit_dur**(-tau_dur)
p2.line(x_fit_dur, y_fit_dur, color='red', line_width=3, line_dash='dashed',
        legend_label=f'Power Law (τ={tau_dur:.2f}, R²={r_dur**2:.3f})')

p2.legend.location = "top_right"

# Arrange plots
layout = row(p1, p2)
show(layout)

print("✓ Interactive power law visualization created")
print(f"\n📊 Size exponent: {tau:.3f} (theory: ~1.5)")
print(f"📊 Duration exponent: {tau_dur:.3f} (theory: ~2.0)")

---

## Part 4: Size-Duration Scaling Relationship

At criticality: **S ∝ D^γ** with γ ≈ 2

---

In [ ]:
# Size-duration relationship
print("Analyzing size-duration relationship...\n")

# Filter valid avalanches
valid_mask = (avalanche_sizes > 0) & (avalanche_durations > 0)
sizes_valid = avalanche_sizes[valid_mask]
durations_valid = avalanche_durations[valid_mask]

# Log-log regression
log_S = np.log10(sizes_valid)
log_D = np.log10(durations_valid)

slope_SD, intercept_SD, r_SD, p_SD, _ = linregress(log_D, log_S)
gamma = slope_SD

print(f"Size-Duration Scaling:")
print(f"  Exponent γ: {gamma:.3f} (theory: ~2.0)")
print(f"  R²: {r_SD**2:.4f}")
print(f"  p-value: {p_SD:.4e}")

# Interactive visualization
p = figure(
    title='Size-Duration Scaling Relationship',
    width=700,
    height=600,
    x_axis_type='log',
    y_axis_type='log',
    x_axis_label='Duration (timesteps)',
    y_axis_label='Size (spikes)',
    toolbar_location='above'
)

# Scatter plot
p.circle(durations_valid, sizes_valid, size=6, alpha=0.4, color='steelblue',
         legend_label='Avalanches')

# Scaling law fit
D_range = np.logspace(np.log10(durations_valid.min()), np.log10(durations_valid.max()), 100)
S_fit = 10**intercept_SD * D_range**gamma
p.line(D_range, S_fit, color='red', line_width=3, line_dash='dashed',
       legend_label=f'S ∝ D^{gamma:.2f} (R²={r_SD**2:.3f})')

p.legend.location = "top_left"

show(p)

print("\n✓ Size-duration scaling visualization created")

---

## Part 5: Branching Parameter Estimation

The branching parameter σ quantifies criticality:
- **σ = # offspring / # parents**
- σ < 1: Subcritical
- σ = 1: Critical ✓
- σ > 1: Supercritical

---

In [ ]:
# Estimate branching parameter
print("Estimating branching parameter...\n")

bp = BranchingProcess()
sigma_estimated = bp.estimate_branching_parameter(critical_activity)

print(f"Branching Parameter Analysis:")
print(f"  Ground truth σ: {sigma_critical:.4f}")
print(f"  Estimated σ: {sigma_estimated:.4f}")
print(f"  Error: {abs(sigma_estimated - sigma_critical):.4f}")
print(f"  Distance from criticality: |σ - 1| = {abs(sigma_estimated - 1):.4f}")

# Classification
if abs(sigma_estimated - 1) < 0.05:
    state = "Critical ✓"
    color = "green"
elif sigma_estimated < 1:
    state = "Subcritical (activity dies)"
    color = "blue"
else:
    state = "Supercritical (activity explodes)"
    color = "red"

print(f"  State: {state}")

---

## Part 6: Test Different Branching Parameters

Compare subcritical, critical, and supercritical regimes.

---

In [ ]:
# Test different σ values
print("Testing different branching parameters...\n")

test_sigmas = [0.7, 0.85, 0.95, 1.0, 1.05, 1.15, 1.3]
results = []

for test_sigma in test_sigmas:
    # Generate activity
    activity = generate_branching_process(n_neurons=200, n_timesteps=3000, sigma=test_sigma)
    
    # Detect avalanches
    avs = detector.detect_avalanches(activity)
    sizes = [av['size'].sum() for av in avs if av['size'].sum() > 0]
    
    # Estimate σ
    sigma_est = bp.estimate_branching_parameter(activity)
    
    results.append({
        'sigma_true': test_sigma,
        'sigma_est': sigma_est,
        'n_avalanches': len(avs),
        'mean_size': np.mean(sizes) if sizes else 0,
        'max_size': np.max(sizes) if sizes else 0,
        'total_activity': activity.sum()
    })
    
    print(f"σ = {test_sigma:.2f}: {len(avs):4d} avalanches, "
          f"mean size = {np.mean(sizes) if sizes else 0:6.1f}, "
          f"max size = {np.max(sizes) if sizes else 0:6.0f}")

results_df = pd.DataFrame(results)

In [ ]:
# Interactive comparison across σ values
print("\nCreating interactive branching parameter comparison...\n")

# Plot 1: Number of avalanches vs σ
p1 = figure(
    title='Avalanche Count vs Branching Parameter',
    width=500,
    height=400,
    x_axis_label='Branching Parameter σ',
    y_axis_label='Number of Avalanches',
    toolbar_location='above'
)

p1.line(results_df['sigma_true'], results_df['n_avalanches'], 
        color='steelblue', line_width=3, alpha=0.8)
p1.circle(results_df['sigma_true'], results_df['n_avalanches'], 
          size=10, color='steelblue', alpha=0.6)

# Mark critical point
critical_line = Span(location=1.0, dimension='height', line_color='red', 
                     line_dash='dashed', line_width=2)
p1.add_layout(critical_line)

# Plot 2: Mean avalanche size vs σ
p2 = figure(
    title='Mean Avalanche Size vs σ',
    width=500,
    height=400,
    x_axis_label='Branching Parameter σ',
    y_axis_label='Mean Size (spikes)',
    toolbar_location='above'
)

p2.line(results_df['sigma_true'], results_df['mean_size'], 
        color='coral', line_width=3, alpha=0.8)
p2.circle(results_df['sigma_true'], results_df['mean_size'], 
          size=10, color='coral', alpha=0.6)

critical_line2 = Span(location=1.0, dimension='height', line_color='red', 
                      line_dash='dashed', line_width=2)
p2.add_layout(critical_line2)

# Plot 3: Max avalanche size vs σ (log scale)
p3 = figure(
    title='Max Avalanche Size vs σ',
    width=500,
    height=400,
    x_axis_label='Branching Parameter σ',
    y_axis_label='Max Size (spikes)',
    y_axis_type='log',
    toolbar_location='above'
)

p3.line(results_df['sigma_true'], results_df['max_size'] + 1,  # +1 for log
        color='purple', line_width=3, alpha=0.8)
p3.circle(results_df['sigma_true'], results_df['max_size'] + 1, 
          size=10, color='purple', alpha=0.6)

critical_line3 = Span(location=1.0, dimension='height', line_color='red', 
                      line_dash='dashed', line_width=2)
p3.add_layout(critical_line3)

# Plot 4: Total activity vs σ
p4 = figure(
    title='Total Activity vs σ',
    width=500,
    height=400,
    x_axis_label='Branching Parameter σ',
    y_axis_label='Total Spikes',
    toolbar_location='above'
)

p4.line(results_df['sigma_true'], results_df['total_activity'], 
        color='green', line_width=3, alpha=0.8)
p4.circle(results_df['sigma_true'], results_df['total_activity'], 
          size=10, color='green', alpha=0.6)

critical_line4 = Span(location=1.0, dimension='height', line_color='red', 
                      line_dash='dashed', line_width=2)
p4.add_layout(critical_line4)

# Arrange plots
layout = gridplot([[p1, p2], [p3, p4]])
show(layout)

print("✓ Interactive branching parameter comparison created")
print("\n📊 Key Insight: Maximum complexity and variability occur at σ ≈ 1 (critical point)")

---

## Part 7: NeuroFMx Criticality Analysis

**Main Analysis**: Test whether NeuroFMx exhibits critical dynamics.

---

In [ ]:
# Create NeuroFMx model
def create_neurofmx_model(device='cpu'):
    """Create NeuroFMx model."""
    if NEUROFMX_AVAILABLE:
        model = NeuroFMX(
            d_model=512,
            n_mamba_blocks=8,
            n_latents=128,
            latent_dim=384,
            n_perceiver_layers=3,
            use_multi_rate=True,
            downsample_rates=[1, 4, 16],
            dropout=0.1
        )
    else:
        # Simplified model
        class SimpleNeuroFMx(nn.Module):
            def __init__(self, d_model=512, n_layers=8):
                super().__init__()
                self.layers = nn.ModuleList([
                    nn.Sequential(
                        nn.Linear(d_model, d_model),
                        nn.GELU(),
                        nn.LayerNorm(d_model)
                    ) for _ in range(n_layers)
                ])
                self.d_model = d_model
                self.n_layers = n_layers
                
            def forward(self, x, return_all_layers=False):
                outputs = [x]
                for layer in self.layers:
                    x = layer(x)
                    outputs.append(x)
                return outputs if return_all_layers else x
        
        model = SimpleNeuroFMx(d_model=512, n_layers=8)
    
    model = model.to(device)
    model.eval()
    return model

# Generate inputs and extract activations
def extract_activations_as_spikes(model, n_samples=100, seq_len=500, device='cpu'):
    """
    Extract model activations and binarize to create 'spike' activity.
    """
    # Generate inputs
    inputs = torch.randn(n_samples, seq_len, model.d_model).to(device)
    
    # Extract activations
    with torch.no_grad():
        if hasattr(model, 'forward') and 'return_all_layers' in model.forward.__code__.co_varnames:
            outputs = model(inputs, return_all_layers=True)
            # Use final layer
            activations = outputs[-1].cpu().numpy()  # (n_samples, seq_len, d_model)
        else:
            activations = model(inputs).cpu().numpy()
    
    # Flatten across samples and features for temporal analysis
    # Shape: (seq_len, n_samples * d_model)
    act_reshaped = activations.transpose(1, 0, 2).reshape(seq_len, -1)
    
    # Binarize: threshold at mean + 1 std
    threshold = act_reshaped.mean() + act_reshaped.std()
    binary_activity = (act_reshaped > threshold).astype(float)
    
    return binary_activity

print("Loading NeuroFMx and extracting activations...\n")
model = create_neurofmx_model(device=device)
n_params = sum(p.numel() for p in model.parameters())

print(f"Model: {type(model).__name__}")
print(f"  Parameters: {n_params:,}")
print(f"  Layers: {model.n_layers if hasattr(model, 'n_layers') else 'N/A'}")

# Extract activations
neurofmx_activity = extract_activations_as_spikes(model, n_samples=50, seq_len=1000, device=device)

print(f"\nExtracted activity:")
print(f"  Shape: {neurofmx_activity.shape}")
print(f"  Total 'spikes': {neurofmx_activity.sum():.0f}")
print(f"  Mean activity: {neurofmx_activity.mean():.4f}")

In [ ]:
# Analyze NeuroFMx criticality
print("Analyzing NeuroFMx for critical dynamics...\n")

# Detect avalanches
neurofmx_avalanches = detector.detect_avalanches(neurofmx_activity)
neurofmx_sizes = np.array([av['size'].sum() for av in neurofmx_avalanches if av['size'].sum() > 0])
neurofmx_durations = np.array([av['duration'] for av in neurofmx_avalanches if av['duration'] > 0])

# Estimate branching parameter
neurofmx_sigma = bp.estimate_branching_parameter(neurofmx_activity)

# Test criticality
crit_detector = CriticalityDetector()
criticality_results = crit_detector.test_criticality(neurofmx_activity)

print(f"NeuroFMx Criticality Results:")
print(f"  Detected avalanches: {len(neurofmx_avalanches)}")
print(f"  Mean size: {neurofmx_sizes.mean():.2f}")
print(f"  Max size: {neurofmx_sizes.max():.0f}")
print(f"  Branching parameter σ: {neurofmx_sigma:.4f}")
print(f"  Distance from criticality: {criticality_results['DCC']:.4f}")
print(f"  Is critical: {criticality_results['is_critical']}")

# Classification
if abs(neurofmx_sigma - 1) < 0.1:
    verdict = "Near-critical ✓"
elif abs(neurofmx_sigma - 1) < 0.2:
    verdict = "Moderately critical"
else:
    verdict = "Far from critical"

print(f"\n📊 Verdict: {verdict}")
print(f"\nInterpretation:")
if abs(neurofmx_sigma - 1) < 0.1:
    print("  NeuroFMx exhibits brain-like critical dynamics!")
    print("  This suggests optimal information processing capacity.")
else:
    print(f"  NeuroFMx is {'subcritical' if neurofmx_sigma < 1 else 'supercritical'}.")
    print("  May benefit from architectural adjustments for criticality.")

In [ ]:
# Comparative visualization: NeuroFMx vs Critical Benchmark
print("Creating comparative criticality visualization...\n")

# Plot 1: Avalanche size distributions
p1 = figure(
    title='Avalanche Size Distribution Comparison',
    width=700,
    height=500,
    x_axis_type='log',
    y_axis_type='log',
    x_axis_label='Avalanche Size',
    y_axis_label='Count',
    toolbar_location='above'
)

# Critical benchmark
unique_sizes_crit, counts_crit = np.unique(sizes_nonzero, return_counts=True)
p1.circle(unique_sizes_crit, counts_crit, size=8, color='gray', alpha=0.6,
          legend_label='Critical Benchmark (σ=1.0)')

# NeuroFMx
unique_sizes_nfmx, counts_nfmx = np.unique(neurofmx_sizes, return_counts=True)
p1.circle(unique_sizes_nfmx, counts_nfmx, size=8, color='steelblue', alpha=0.6,
          legend_label=f'NeuroFMx (σ={neurofmx_sigma:.3f})')

p1.legend.location = "top_right"

# Plot 2: Branching parameter comparison
p2 = figure(
    title='Branching Parameter Comparison',
    width=500,
    height=500,
    x_range=['Critical\nBenchmark', 'NeuroFMx'],
    y_range=[0.5, 1.5],
    y_axis_label='Branching Parameter σ',
    toolbar_location='above'
)

models = ['Critical\nBenchmark', 'NeuroFMx']
sigmas = [sigma_estimated, neurofmx_sigma]
colors_sigma = ['gray', 'steelblue']

p2.vbar(x=models, top=sigmas, width=0.6, color=colors_sigma, alpha=0.7)

# Critical line
critical_span = Span(location=1.0, dimension='width', line_color='red', 
                     line_dash='dashed', line_width=3)
p2.add_layout(critical_span)

# Tolerance band
from bokeh.models import BoxAnnotation
tolerance = BoxAnnotation(bottom=0.9, top=1.1, fill_alpha=0.1, fill_color='green')
p2.add_layout(tolerance)

# Arrange plots
layout = row(p1, p2)
show(layout)

print("✓ Comparative criticality visualization created")

---

## Part 8: NeuroFMx Architecture & Criticality

**Why NeuroFMx May Exhibit Critical Dynamics**:

### Architectural Features Promoting Criticality

1. **State-Space Models (Mamba)**
   - Continuous-time dynamics with feedback
   - Similar to recurrent neural networks that can self-organize to criticality
   - Selective state propagation acts as adaptive pruning

2. **Multi-Rate Processing**
   - 1x, 4x, 16x downsampling creates temporal hierarchy
   - Cross-scale coupling enables avalanche propagation across timescales
   - Analogous to cortical hierarchy (V1 → IT)

3. **Layer Normalization**
   - Regulates activity levels (prevents explosion/collapse)
   - Acts as homeostatic mechanism (like biological neurons)
   - Maintains network in operational regime

4. **Perceiver-IO Compression**
   - Bottleneck creates information flow constraints
   - Selective attention may create power-law statistics
   - Similar to attentional bottleneck in brain

### Comparison to Biological Criticality

| Feature | Biological Brain | NeuroFMx |
|---------|------------------|----------|
| Feedback loops | Recurrent connectivity | State-space feedback |
| Homeostasis | Synaptic plasticity | Layer normalization |
| Hierarchy | Cortical layers | Multi-rate streams |
| Bottleneck | Thalamus, attention | Perceiver latents |

### Implications

If NeuroFMx is near-critical:
- ✓ **Optimal information processing**
- ✓ **Maximal dynamic range**
- ✓ **Brain-like computation**
- ✓ **Emergent complexity**

---

---

## Conclusion

This notebook demonstrated:

1. **Avalanche Detection**: Identified cascading activity patterns
2. **Power Law Analysis**: Tested for scale-free size/duration distributions
3. **Branching Parameter**: Quantified distance from critical point
4. **σ Comparison**: Explored subcritical, critical, and supercritical regimes
5. **NeuroFMx Testing**: Analyzed foundation model for critical dynamics
6. **Architecture Analysis**: Connected design features to criticality

**Key Takeaways**:
- Criticality (σ ≈ 1) enables optimal neural computation
- NeuroFMx architecture includes features that promote criticality
- Power laws and avalanches indicate brain-like organization
- Multi-rate processing and normalization support critical regime

**Next Steps**:
- **Notebook 22**: Multifractal analysis - quantify temporal complexity beyond power laws
- **Training Analysis**: Track criticality emergence during NeuroFMx training
- **Interventions**: Test whether architectural changes affect criticality

---